## Run statistics on the splits

In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split

from src.zoorvival.data import load_tcga_data
from src.zoorvival.nn.training import as_torch_dataset

%reload_ext autoreload
%autoreload 2

np.random.seed(42)

In [2]:
PROJECT = "BLCA"
signatures = 'other'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
def request_tcga_file_info(data_type: str, project_ids=None) -> pd.DataFrame:
    """
    Get TCGA file information from GDC API.

    Args:
        data_type (str): The type of data to retrieve.
        project_ids (list[str], optional): List of project IDs to filter by. Defaults to None.
    
    Returns:
        pd.DataFrame: DataFrame containing file information.
    """
    if project_ids is None:
        project_ids = get_tcga_projects()
    fields = [
        "file_name",
        "md5sum",
        "cases.submitter_id",
        "cases.samples.sample_type",
        "cases.project.project_id",
        "cases.project.primary_site",
    ]
    fields = ",".join(fields)
    files_endpt = "https://api.gdc.cancer.gov/files"
    filters = {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {
                    "field": "files.experimental_strategy",
                    "value": [data_type],
                },
            },
            {
                "op": "in",
                "content": {
                    "field": "cases.project.project_id",
                    "value": project_ids,
                },
            },
        ],
    }
    params = {"filters": filters, "fields": fields, "format": "TSV", "size": "200000"}
    response = requests.post(
        files_endpt, headers={"Content-Type": "application/json"}, json=params
    )
    return pd.read_csv(StringIO(response.content.decode("utf-8")), sep="\t")


def process_tcga_file_info(df_files, suffix, data_type) -> pd.DataFrame:
    rename_cols = {
        "cases.0.project.project_id": "project_id",
        "cases.0.project.primary_site": "primary_site",
        "cases.0.submitter_id": "submitter_id",
        "cases.0.samples.0.sample_type": "sample_type",
        "file_name": f"{data_type}_file_name",
        "id": f"{data_type}_file_id",
        "md5sum": f"{data_type}_md5sum",
    }
    df_files = df_files.rename(columns=rename_cols)
    df_files = df_files[df_files[f"{data_type}_file_name"].str.endswith(suffix)]
    df_files = df_files[df_files["sample_type"] == "Primary Tumor"]
    df_files = df_files[~df_files.duplicated(subset=["submitter_id"], keep="first")]
    return df_files


def get_suffix_counts(df_files) -> pd.Series:
    file_col = [c for c in df_files.columns if "file_name" in c][0]
    file_suffixes = [".".join(f.split(".")[-2:]) for f in df_files[file_col]]
    suffix_counts = pd.Series(file_suffixes).value_counts()
    suffix_counts = suffix_counts[suffix_counts > 1]
    suffix_counts = suffix_counts.reset_index()
    return suffix_counts

def get_tcga_projects() -> list[str]:
    """
    Get a list of TCGA projects from the GDC API.
    """
    response = requests.get("https://api.gdc.cancer.gov/projects", params={"size": 10000})
    response.raise_for_status()
    projects = response.json()["data"]["hits"]
    projects = [proj for proj in projects if proj.get("project_id", "").startswith("TCGA-")]
    projects = [proj["id"] for proj in projects]
    return projects

### Start:

In [4]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from random import shuffle
from functools import reduce
import copy
import torch
from torch.utils.data import DataLoader, random_split
import builtins
set = builtins.set

from src.zoorvival.data import load_tcga_data
from src.zoorvival.nn.training import as_torch_dataset

%reload_ext autoreload
%autoreload 2

np.random.seed(42)

In [5]:
censorship_var = 'censorship_dss'
label_col = 'survival_months_dss'
if signatures in ['other']:
    censorship_var = 'censorship_os'
    label_col = 'survival_months_os'

In [6]:
# get the whole data
study = 'BLCA'

if signatures in ['other']:
    omics_data_c = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/other/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )
    omics_data_h = omics_data_c
    omics_data_x = omics_data_c

else:
    # omics data
    omics_data_c = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/combine/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

    omics_data_h = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/hallmarks/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

    omics_data_x = pd.read_csv(
        os.path.join(f'src/datasets_csv/raw_rna_data/xena/{study}', 'rna_clean.csv'),    # all rna data
        engine='python', 
        index_col=0
    )

# metadata
label_data = pd.read_csv(f'src/datasets_csv/metadata/tcga_{study}.csv', low_memory=False)    # contains pairing between case_id and slide_id(s) + info on survival

# clinical data
clinical_data = pd.read_csv(f'src/datasets_csv/clinical_data/tcga_{study}_clinical.csv', index_col=0) # clinical data


In [7]:
db = load_tcga_data(PROJECT)

In [8]:
db.train.df_clinical.index

Index(['HQ-A2OF', 'ZF-A9RD', 'BT-A20P', 'BT-A20W', 'GD-A6C6', 'UY-A9PA',
       'GV-A40E', 'FD-A3B8', 'GU-AATO', 'XF-A9T8',
       ...
       'S5-AA26', 'XF-AAMZ', 'CU-A0YO', 'CF-A3MI', 'GV-A6ZA', 'DK-A2I4',
       'GU-A42R', 'K4-A6FZ', 'XF-A9SH', 'FJ-A3Z7'],
      dtype='object', length=295)

In [9]:
train_patients = list('TCGA-' + i for i in db.train.df_clinical.index.astype(str))
test_patients = list('TCGA-' + i for i in db.test.df_clinical.index.astype(str))

train_patients.extend(test_patients)
wsi_patients = train_patients
wsi_patients = list(set(wsi_patients))
print('number of wsi present in wsi database: ', len(wsi_patients))

number of wsi present in wsi database:  369


In [10]:
# get data

def get_client_data(split_num, num_clients, set_i, q_bins):

    clients_0 = []

    for i in range(num_clients):
        client = {}
        path_ids = f'src/splits_{study}/split_' + str(split_num) + '/client_' + str(i) + '/train.csv'
        client['ids'] = pd.read_csv(path_ids)
        split = client['ids'][set_i].values

        client['omics'] = omics_data_x[omics_data_x.index.isin(split)]
        #print('client ', i, 'omics: \n', client['omics'].head(3))

        client['labels'] = label_data[label_data['case_id'].isin(split)]
        #print('labels: \n', client['labels'].head(3))

        client['clinical'] = clinical_data[clinical_data['case_id'].isin(split)]
        #print('clinical: \n', client['clinical'].head(3))

        #patients_df = client['labels'].drop_duplicates(['case_id']).copy()
        patients_df = client['labels'].copy()

        disc_labels, q_bins = pd.cut(patients_df[label_col], bins=q_bins, retbins=True, labels=False, right=False, include_lowest=True)
        # returns An array-like object representing the respective bin for each value of x; and The computed or specified bins. For scalar or sequence bins, this is an ndarray with the computed bins
        patients_df.insert(2, 'label', disc_labels.values.astype(int))
        client['labels'].insert(2, 'label', disc_labels.values.astype(int))
        client['labels'] = client['labels'].drop_duplicates(['case_id']).copy()
        client['bins'] = q_bins
        #print('discretized labels: \n', client['labels'].head(3))

        clients_0.append(client)
    
    return clients_0



In [11]:
# run statistics
def run_statistics_on_clients(all_splits_clients, set_i, num_clients, split_num):

    for client_id in range(num_clients):
        print('CLIENT ', client_id)
        all_event_times = []
        all_censorships = []

        client = all_splits_clients[split_num][client_id]
        # drop duplicates for multiple case_ids in clients
        #client = client['labels'].drop_duplicates(['case_id'])

        event_times = client['labels'][label_col].values
        censorships = client['labels'][censorship_var].values

        all_event_times = np.array(event_times, dtype=float)
        all_censorships = np.array(censorships, dtype=int)

        print(f'Client {client_id} - {set_i} set statistics:')
        print('Number of samples:', len(all_event_times))
        print('Event times - mean:', np.mean(all_event_times), ', median:', np.median(all_event_times), ', std:', np.std(all_event_times))
        print('Censorships - num censored:', np.sum(all_censorships==1), ', num events:', np.sum(all_censorships==0))
        if len(all_censorships) > 0:
            #print('something is wrong with the division')
            print('Censorship rate:', np.sum(all_censorships==1)/len(all_censorships))
        else: 
            print('all_censorship has lenght 0')
        
        print('discretized labels distribution:')
        print(client['labels']['label'].value_counts().sort_index())

        print('discretization considering the censorship')
        label_dict = {}
        key_count = 0
        for i in range(len(client['bins'])-1):  # obtain label_dict = {(0-5,0) : 0; (0-5,1): 1; (5-10,0): 2; ...}? to len(self_bins -2)
            for c in [0, 1]:
                label_dict.update({(i, c):key_count})
                key_count+=1
                
        label = copy.deepcopy(client['labels'])
        for i in label.index: # for each sample, i is the index
            key = label.loc[i, 'label'] 
            label.at[i, 'disc_label'] = key    # Access a single value for a row/column label pair.
            censorship = label.loc[i, censorship_var]
            key = (key, int(censorship))
            label.at[i, 'label'] = label_dict[key]  # put the corresponding key_count in 'label' of label_data; give class for each bin-censorship option
        
        print(label['label'].value_counts().sort_index())


        print('Wsis missing:')
        new_label = label.drop_duplicates(['case_id'])
        num_wsi = 0
        num_wsi_missing = 0
        missing_ids = []
        for i in new_label['case_id']:
            if i in wsi_patients:
                num_wsi += 1
            else:
                num_wsi_missing +=1
                missing_ids.append(i)

        print('number patients with wsi: ', num_wsi)
        print('number patients without wsi: ', num_wsi_missing)

        print()
        print('Are the missing patients events')
        missing_labels = label[label['case_id'].isin(missing_ids)]
        print(missing_labels['label'].value_counts().sort_index())

        
        print('---')

In [12]:
# consider train set
set_i = 'train'
num_clients = 3
num_splits = 5

# discretize as in the code
patients_df_all = label_data.drop_duplicates(['case_id']).copy()
uncensored_df_whole = patients_df_all[patients_df_all[censorship_var] < 1]

disc_labels, q_bins = pd.qcut(uncensored_df_whole[label_col], q=4, retbins=True, labels=False)
q_bins[-1] = label_data[label_col].max() + 1e-6
q_bins[0] = label_data[label_col].min() - 1e-6
print('quantile bins:', q_bins)

# get all train data from all splits
all_splits_clients = []
for split_num in range(num_splits):
    clients = get_client_data(split_num, num_clients, set_i, q_bins)
    all_splits_clients.append(clients)


# run statistics on all splits clients
for split_num in range(num_splits):
    print('SPLIT ', split_num)
    run_statistics_on_clients(all_splits_clients, set_i, num_clients, split_num)

quantile bins: [ -2.100001   7.88      13.65      22.0675   163.170001]
SPLIT  0
CLIENT  0
Client 0 - train set statistics:
Number of samples: 89
Event times - mean: 23.583483146067415 , median: 17.15 , std: 21.539935948759975
Censorships - num censored: 53 , num events: 36
Censorship rate: 0.5955056179775281
discretized labels distribution:
label
0    14
1    17
2    28
3    30
Name: count, dtype: int64
discretization considering the censorship
label
0     9
1     5
2     9
3     8
4    11
5    17
6     7
7    23
Name: count, dtype: int64
Wsis missing:
number patients with wsi:  88
number patients without wsi:  1

Are the missing patients events
label
1    1
Name: count, dtype: int64
---
CLIENT  1
Client 1 - train set statistics:
Number of samples: 89
Event times - mean: 26.520000000000003 , median: 17.41 , std: 26.926620010104113
Censorships - num censored: 41 , num events: 48
Censorship rate: 0.4606741573033708
discretized labels distribution:
label
0    12
1    21
2    21
3    35
N

In [13]:
# consider train set
set_i = 'val'
num_clients = 1
num_splits = 5

# discretize as in the code
patients_df_all = label_data.drop_duplicates(['case_id']).copy()
uncensored_df_whole = patients_df_all[patients_df_all[censorship_var] < 1]

disc_labels, q_bins = pd.qcut(uncensored_df_whole[label_col], q=4, retbins=True, labels=False)
q_bins[-1] = label_data[label_col].max() + 1e-6
q_bins[0] = label_data[label_col].min() - 1e-6
print('quantile bins:', q_bins)

# get all train data from all splits
all_splits_clients = []
for split_num in range(num_splits):
    
    clients = []
    for i in range(num_clients):
        client = {}
        path_ids = f'src/splits_{study}/split_' + str(split_num) + '/val.csv'
        client['ids'] = pd.read_csv(path_ids)
        split = client['ids'][set_i].values

        client['labels'] = label_data[label_data['case_id'].isin(split)]
        #print('labels: \n', client['labels'].head(3))

        patients_df = client['labels'].copy()

        disc_labels, q_bins = pd.cut(patients_df[label_col], bins=q_bins, retbins=True, labels=False, right=False, include_lowest=True)
        # returns An array-like object representing the respective bin for each value of x; and The computed or specified bins. For scalar or sequence bins, this is an ndarray with the computed bins
        patients_df.insert(2, 'label', disc_labels.values.astype(int))
        client['labels'].insert(2, 'label', disc_labels.values.astype(int))
        client['bins'] = q_bins
        #print('discretized labels: \n', client['labels'].head(3))

        clients.append(client)
    
    print('clients length:', len(clients))
    all_splits_clients.append(clients)


# run statistics on all splits clients
for split_num in range(num_splits):
    print('SPLIT ', split_num)
    run_statistics_on_clients(all_splits_clients, set_i, num_clients, split_num)

quantile bins: [ -2.100001   7.88      13.65      22.0675   163.170001]
clients length: 1
clients length: 1
clients length: 1
clients length: 1
clients length: 1
SPLIT  0
CLIENT  0
Client 0 - val set statistics:
Number of samples: 48
Event times - mean: 23.571250000000003 , median: 18.805 , std: 16.94724837854708
Censorships - num censored: 26 , num events: 22
Censorship rate: 0.5416666666666666
discretized labels distribution:
label
0     5
1    10
2    13
3    20
Name: count, dtype: int64
discretization considering the censorship
label
0     3
1     2
2     6
3     4
4     5
5     8
6     8
7    12
Name: count, dtype: int64
Wsis missing:
number patients with wsi:  48
number patients without wsi:  0

Are the missing patients events
Series([], Name: count, dtype: int64)
---
SPLIT  1
CLIENT  0
Client 0 - val set statistics:
Number of samples: 48
Event times - mean: 27.08270833333333 , median: 18.085 , std: 24.981851057282842
Censorships - num censored: 26 , num events: 22
Censorship rat

In [14]:
# consider train set
set_i = 'test'
num_clients = 1
num_splits = 5

# discretize as in the code
patients_df_all = label_data.drop_duplicates(['case_id']).copy()
uncensored_df_whole = patients_df_all[patients_df_all[censorship_var] < 1]

disc_labels, q_bins = pd.qcut(uncensored_df_whole[label_col], q=4, retbins=True, labels=False)
q_bins[-1] = label_data[label_col].max() + 1e-6
q_bins[0] = label_data[label_col].min() - 1e-6
print('quantile bins:', q_bins)

# get all train data from all splits
all_splits_clients = []
for split_num in range(num_splits):
    
    clients = []
    for i in range(num_clients):
        client = {}
        path_ids = f'src/splits_{study}/split_' + str(split_num) + '/test.csv'
        client['ids'] = pd.read_csv(path_ids)
        split = client['ids'][set_i].values

        client['labels'] = label_data[label_data['case_id'].isin(split)]
        #print('labels: \n', client['labels'].head(3))

        patients_df = client['labels'].copy()

        disc_labels, q_bins = pd.cut(patients_df[label_col], bins=q_bins, retbins=True, labels=False, right=False, include_lowest=True)
        # returns An array-like object representing the respective bin for each value of x; and The computed or specified bins. For scalar or sequence bins, this is an ndarray with the computed bins
        patients_df.insert(2, 'label', disc_labels.values.astype(int))
        client['labels'].insert(2, 'label', disc_labels.values.astype(int))
        client['bins'] = q_bins
        #print('discretized labels: \n', client['labels'].head(3))

        clients.append(client)
    
    print('clients length:', len(clients))
    all_splits_clients.append(clients)


# run statistics on all splits clients
for split_num in range(num_splits):
    print('SPLIT ', split_num)
    run_statistics_on_clients(all_splits_clients, set_i, num_clients, split_num)

quantile bins: [ -2.100001   7.88      13.65      22.0675   163.170001]
clients length: 1
clients length: 1
clients length: 1
clients length: 1
clients length: 1
SPLIT  0
CLIENT  0
Client 0 - test set statistics:
Number of samples: 56
Event times - mean: 32.69857142857143 , median: 18.725 , std: 34.20689061894294
Censorships - num censored: 31 , num events: 25
Censorship rate: 0.5535714285714286
discretized labels distribution:
label
0    11
1    10
2    15
3    20
Name: count, dtype: int64
discretization considering the censorship
label
0     8
1     3
2     4
3     6
4     6
5     9
6     7
7    13
Name: count, dtype: int64
Wsis missing:
number patients with wsi:  56
number patients without wsi:  0

Are the missing patients events
Series([], Name: count, dtype: int64)
---
SPLIT  1
CLIENT  0
Client 0 - test set statistics:
Number of samples: 56
Event times - mean: 24.28928571428571 , median: 15.850000000000001 , std: 25.084642773243242
Censorships - num censored: 31 , num events: 25
C